# Gerador de Poesias — Álvaro de Campos

Fine-tuning de **Llama 3.2-1B** com **LoRA** para gerar poemas no estilo de Álvaro de Campos.

**Melhorias em relação ao treino local (Apple Silicon MPS):**
- GPU Colab (T4/A100): ~20× mais rápida que Apple Silicon MPS
- LoRA rank=16, alpha=32 com módulos de atenção **e** FFN — maior capacidade de adaptação
- Gradient clipping (`max_grad_norm=1.0`) — elimina spikes de gradiente
- Gradient checkpointing — poupa ~35% de VRAM
- 3 épocas (o treino local mostrou que a convergência ocorre na época 3)
- `warmup_steps=50` fixo — igual ao treino local, garante estabilidade inicial
- Dataset limpo: nomes de autores e aspas tipográficas removidos via spaCy NER
- `bad_words_ids` bloqueia geração de nomes dos autores na inferência
- `no_repeat_ngram_size=3` melhora coerência textual

> **Antes de começar:** vá a *Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4 ou A100)*

## 0. Instalação de dependências


In [1]:
%%capture
!pip install transformers>=4.45.0 peft>=0.10.0 accelerate>=0.29.0 \
            datasets>=2.18.0 bitsandbytes>=0.43.0 \
            spacy huggingface_hub
!python -m spacy download pt_core_news_sm

##1. Importação de dependências


In [2]:
# ── Biblioteca padrão ────────────────────────────────────────────────────────
import csv        # leitura do dataset CSV                          → sec. 5
import io         # buffer em memória para download via Drive API   → sec. 4
import math       # cálculo de perplexidade: exp(eval_loss)        → sec. 9
import os         # caminhos, criação de pastas, listagem          → sec. 4, 11
import re         # limpeza de texto (espaços, linhas)             → sec. 5
import shutil     # cópia do ficheiro CSV do Drive montado         → sec. 4

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch      # detecção de GPU/dtype; inferência              → sec. 1, 7, 10

# ── Google Colab ─────────────────────────────────────────────────────────────
from google.colab import drive      # montar o Google Drive        → sec. 4
from google.colab import userdata   # ler o token HF dos Secrets   → sec. 3a

# ── Hugging Face Hub ──────────────────────────────────────────────────────────
from huggingface_hub import login   # autenticação no HF Hub       → sec. 3a, 12

# ── Transformers ──────────────────────────────────────────────────────────────
from transformers import (
    AutoTokenizer,                  # tokenizador do Llama          → sec. 6
    AutoModelForCausalLM,           # modelo base Llama 3.2-1B     → sec. 7
    TrainingArguments,              # hiperparâmetros do treino     → sec. 8
    Trainer,                        # ciclo de treino/avaliação     → sec. 8, 9
    DataCollatorForLanguageModeling,# collator para LM causal       → sec. 8
)

# ── PEFT (LoRA) ───────────────────────────────────────────────────────────────
from peft import (
    LoraConfig,     # configuração do adaptador LoRA               → sec. 7
    get_peft_model, # aplica LoRA ao modelo base                   → sec. 7
    TaskType,       # especifica tarefa: CAUSAL_LM                 → sec. 7
)

# ── Datasets ──────────────────────────────────────────────────────────────────
from datasets import Dataset  # converte lista de textos em dataset HF → sec. 6

# ── spaCy ─────────────────────────────────────────────────────────────────────
import spacy  # NER para remover nomes de autores do dataset       → sec. 5
              # modelo: pt_core_news_sm (português de Portugal)

print('Imports carregados com sucesso.')

Imports carregados com sucesso.


## 2. Verificar GPU e Ambiente

In [3]:
print('=== Ambiente Colab ===')
if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16 = torch.cuda.is_bf16_supported()
    print(f'GPU             : {gpu}')
    print(f'Memória GPU     : {mem:.1f} GB')
    print(f'CUDA            : {torch.version.cuda}')
    print(f'BFloat16        : {bf16}')
    DTYPE = torch.bfloat16 if bf16 else torch.float16
    DTYPE_STR = 'bfloat16' if bf16 else 'float16'
else:
    print('AVISO: GPU nao detectada — treino sera muito lento na CPU!')
    DTYPE = torch.float32
    DTYPE_STR = 'float32'

print(f'Dtype utilizado : {DTYPE_STR}')

=== Ambiente Colab ===
GPU             : Tesla T4
Memória GPU     : 15.6 GB
CUDA            : 12.8
BFloat16        : True
Dtype utilizado : bfloat16


## 3. Configuração Central

In [4]:
# =============================================================
# hiperparâmetros
# =============================================================

# Modelo base (requer token HF e acesso aprovado ao Llama)
MODEL_NAME   = 'meta-llama/Llama-3.2-1B'

# Caminhos
DATASET_PATH = '/content/dataset_alvaro_campos.csv'
OUTPUT_DIR   = '/content/drive/MyDrive/modelo_poesia_colab'

# LoRA
LORA_R       = 16    # rank (8 no treino anterior; 16 captura mais nuances)
LORA_ALPHA   = 32    # escala (regra prática: alpha = 2 × r)
LORA_DROPOUT = 0.05

# Treino
EPOCHS        = 3       # convergência ocorre na época 3
BATCH_SIZE    = 2       # igual ao treino local — mantém batch efectivo = 8
GRAD_ACCUM    = 4       # batch efectivo = BATCH_SIZE × GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS  = 50      # fixo (igual ao treino local; ratio dinâmico era ~10 steps — insuficiente)
MAX_GRAD_NORM = 1.0     # gradient clipping
MAX_LENGTH    = 768     # igual ao treino local

# Inferência
GEN_TEMPERATURE  = 0.75
GEN_TOP_P        = 0.90
GEN_TOP_K        = 40
GEN_REP_PENALTY  = 1.3
GEN_NO_REPEAT_N  = 3
GEN_MAX_TOKENS   = 200

print('Configuração carregada.')

Configuração carregada.


## 4. Google Drive e Dataset

In [5]:
# Monta o Drive para guardar o modelo treinado
drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Pasta de saída: {OUTPUT_DIR}')

# ── Localiza o dataset no Drive montado ──────────────────────────────────────
# O ficheiro é procurado por nome em todo o MyDrive.
# Não são necessárias permissões adicionais — usa o Drive já autenticado.
DATASET_NAME = 'dataset_alvaro_campos.csv'
DATASET_PATH = f'/content/{DATASET_NAME}'

origem = None
for root, _, files in os.walk('/content/drive/MyDrive'):
    if DATASET_NAME in files:
        origem = os.path.join(root, DATASET_NAME)
        break

if origem:
    shutil.copy(origem, DATASET_PATH)
    print(f'Dataset copiado de : {origem}')
    print(f'Disponível em      : {DATASET_PATH} ({os.path.getsize(DATASET_PATH):,} bytes)')
else:
    print(f'ERRO: "{DATASET_NAME}" não encontrado no MyDrive.')
    print('Verifique se o ficheiro está partilhado consigo e visível no Drive.')

Mounted at /content/drive
Pasta de saída: /content/drive/MyDrive/modelo_poesia_colab
Dataset copiado de : /content/drive/MyDrive/Colab Notebooks/dataset_alvaro_campos.csv
Disponível em      : /content/dataset_alvaro_campos.csv (389,497 bytes)


## 5. Preparação do Dataset

In [6]:
# Modelo spaCy para português de Portugal
nlp = spacy.load('pt_core_news_sm', disable=['parser', 'lemmatizer'])

# Nomes a remover dos textos do dataset
NOMES_PROIBIDOS = [
    'Fernando Pessoa', 'Alvaro de Campos', 'Álvaro de Campos',
    'Bernardo Soares', 'Alberto Caeiro', 'Ricardo Reis',
]


def limpar_texto(texto):
    # Remove nomes de autores
    for nome in NOMES_PROIBIDOS:
        texto = texto.replace(nome, '')

    # Remove aspas tipográficas ("" '' «»)
    for char in ['\u201c', '\u201d', '\u2018', '\u2019', '\u00ab', '\u00bb']:
        texto = texto.replace(char, '')

    # Remove entidades PERSON detectadas por NER (spaCy pt_core_news_sm)
    doc = nlp(texto[:1000])  # limita a 1000 chars para velocidade
    for ent in doc.ents:
        if ent.label_ == 'PER':
            texto = texto.replace(ent.text, '')

    # Normaliza espaços e linhas em branco excessivas
    texto = re.sub(r' {2,}', ' ', texto)
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto.strip()


def carregar_e_limpar(path):
    textos = []
    n_vazios = n_curtos = 0

    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        if 'texto' not in (reader.fieldnames or []):
            raise ValueError(f'Coluna "texto" nao encontrada. Colunas: {reader.fieldnames}')

        for row in reader:
            texto = (row.get('texto') or '').strip()
            if not texto:
                n_vazios += 1
                continue
            if len(texto) < 50:
                n_curtos += 1
                continue
            textos.append(limpar_texto(texto))

    print(f'Poemas carregados   : {len(textos)}')
    print(f'Ignorados (vazios)  : {n_vazios}')
    print(f'Ignorados (curtos)  : {n_curtos}')
    return textos


TEXTOS = carregar_e_limpar(DATASET_PATH)

# Estatísticas básicas
comprimentos = [len(t) for t in TEXTOS]
print(f'\nChars — mediana: {sorted(comprimentos)[len(comprimentos)//2]}')
print(f'Chars — min/max: {min(comprimentos)} / {max(comprimentos)}')
print('\n--- Exemplo de poema após limpeza ---')
print(TEXTOS[0][:400])

Poemas carregados   : 314
Ignorados (vazios)  : 0
Ignorados (curtos)  : 0

Chars — mediana: 652
Chars — min/max: 67 / 43435

--- Exemplo de poema após limpeza ---
ANIVERSÁRIO

No tempo em que festejavam o dia dos meus anos,
Eu era feliz e ninguém estava morto.
Na casa antiga, até eu fazer anos era uma tradição de há séculos,
E a alegria de todos, e a minha, estava certa com uma religião qualquer.
No tempo em que festejavam o dia dos meus anos,
Eu tinha a grande saúde de não perceber coisa nenhuma,
De ser inteligente para entre a família,
E de não ter as esp


## 6. Tokenização

In [7]:
print(f'A carregar tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print(f'Vocabulário: {len(tokenizer)} tokens')


def tokenizar(textos):
    # Cada poema termina com EOS para o modelo aprender a delimitar o fim
    textos_com_eos = [f'{t}{tokenizer.eos_token}' for t in textos]

    def tokenize_fn(batch):
        return tokenizer(
            batch['texto'],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

    ds = Dataset.from_dict({'texto': textos_com_eos})
    return ds.map(tokenize_fn, batched=True, remove_columns=['texto'])


dataset  = tokenizar(TEXTOS)
split    = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds   = split['test']

print(f'\nTreino:    {len(train_ds)} poemas')
print(f'Validação: {len(val_ds)} poemas')

A carregar tokenizer: meta-llama/Llama-3.2-1B


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Vocabulário: 128256 tokens


Map:   0%|          | 0/314 [00:00<?, ? examples/s]


Treino:    282 poemas
Validação: 32 poemas


## 7. Modelo Base + LoRA

In [8]:
print(f'A carregar modelo: {MODEL_NAME}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map='auto',
)
model.config.use_cache = False

# Gradient checkpointing: economiza ~35% de VRAM
model.gradient_checkpointing_enable()

# LoRA rank=16 com módulos de atenção + FFN (gate/up/down proj do Llama)
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

A carregar modelo: meta-llama/Llama-3.2-1B


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


## 8. Fine-Tuning

In [9]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=(DTYPE_STR == 'bfloat16'),
    fp16=(DTYPE_STR == 'float16'),
    report_to='none',
    optim='adamw_torch',
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print('Treinando...')
trainer.train()
print('Treino concluído!')

Treinando...


Epoch,Training Loss,Validation Loss
1,3.327578,3.435790
2,3.005451,3.359629
3,2.822626,3.387053


Treino concluído!


## 9. Avaliação — Perplexidade

In [10]:
print('A avaliar no conjunto de validação...')
results      = trainer.evaluate()
eval_loss    = results['eval_loss']
perplexidade = math.exp(eval_loss)

print(f'Eval Loss    : {eval_loss:.4f}')
print(f'Perplexidade : {perplexidade:.2f}')
print()
print('Referência:')
print('  < 15  → excelente aderência ao estilo')
print('  15-20 → boa aderência')
print('  > 20  → aderência moderada (treino local: 27.85)')

A avaliar no conjunto de validação...


Eval Loss    : 3.3596
Perplexidade : 28.78

Referência:
  < 15  → excelente aderência ao estilo
  15-20 → boa aderência
  > 20  → aderência moderada (treino local: 27.85)


## 10. Geração de Poemas

In [11]:
def get_bad_words_ids(tokenizer):
    frases_proibidas = [
        'Fernando Pessoa', 'Alvaro de Campos', 'Álvaro de Campos',
        'Bernardo Soares', 'Alberto Caeiro', 'Ricardo Reis',
        '\u201c', '\u201d', '\u2018', '\u2019', '\u00ab', '\u00bb',
    ]
    bad = []
    seen = set()
    for frase in frases_proibidas:
        for prefixo in ['', ' ']:
            ids = tuple(tokenizer.encode(prefixo + frase, add_special_tokens=False))
            if ids and ids not in seen:
                bad.append(list(ids))
                seen.add(ids)
    return bad


BAD_WORDS_IDS = get_bad_words_ids(tokenizer)


def gerar_poema(verso_inicial, max_new_tokens=GEN_MAX_TOKENS):
    model.eval()
    inputs = tokenizer(verso_inicial, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            top_k=GEN_TOP_K,
            repetition_penalty=GEN_REP_PENALTY,
            no_repeat_ngram_size=GEN_NO_REPEAT_N,
            bad_words_ids=BAD_WORDS_IDS,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


print(f'Gerador pronto. Sequências bloqueadas: {len(BAD_WORDS_IDS)}')

Gerador pronto. Sequências bloqueadas: 24


In [12]:
versos_teste = [
    'Estou cansado de tudo,',
    'O mar imenso e indiferente,',
    'As maquinas trabalham enquanto eu',
    'Sinto em mim mil vidas a perderem-se',
    'Lisboa, a cidade das saudades,',
]

for verso in versos_teste:
    print(f'Verso inicial: {verso}')
    print('=' * 55)
    print(gerar_poema(verso))
    print()

Verso inicial: Estou cansado de tudo,
Estou cansado de tudo, da vida em particular e do universo comum.

Meu Deus! O que temos para ser feliz!
Não é certo estar aqui,
Nem nem lá fora.
Eu estaria mais contente se tivesse a consciência
De não ter nenhuma coisa...
Mas como isso me parece?
Pobre pavorista das emoções!
Que pena no mundo dos humanos haver talvez o sono naturalmente normal.

Verso inicial: O mar imenso e indiferente,
O mar imenso e indiferente, a terra que passa,
A cidade onde o trabalho é só um ruído de pedras
E as pessoas são como os objetos dos escritos;
À noite em silêncio da rua — eu olho para trás —
É na luz do pôr-do-sol nas casas ao longe.
No fundo brilha alguma coisa com a lua...
Que grande casa ou edifício distante!
Mas quem sabe se ela não seja também uma das minhas?
Seja isto sempre mesmo. Deixem-me ir!...
Pobre gente humana viva sem vida nem alma...!
Não tenha pena nenhuma: sinto-a por mim próprio:
São todos humanos assim; morrem-se igualmente..."
De repouso fina

## 11. Guardar Modelo no Drive

In [13]:
print(f'A guardar em: {OUTPUT_DIR}')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

ficheiros = os.listdir(OUTPUT_DIR)
print(f'Ficheiros guardados: {ficheiros}')
print('Modelo guardado com sucesso no Google Drive!')

A guardar em: /content/drive/MyDrive/modelo_poesia_colab
Ficheiros guardados: ['checkpoint-72', 'checkpoint-108', 'README.md', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'tokenizer.json']
Modelo guardado com sucesso no Google Drive!


## 12. Publicar no Hugging Face Hub

Obtenha o seu token em https://huggingface.co/settings/tokens

In [14]:
HF_TOKEN   = ''     # cole aqui o seu token (write access)
HF_REPO_ID = 'SEU_UTILIZADOR/gerador-poesias-alvaro-campos'

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print(f'Modelo publicado: https://huggingface.co/{HF_REPO_ID}')
else:
    print('HF_TOKEN nao definido — passo ignorado.')

HF_TOKEN nao definido — passo ignorado.
